<a href="https://colab.research.google.com/github/SamaelIncubus/AI/blob/main/40768110_Part_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment Template

Please follow the template below to submit part A of your assignment. Make sure all the outputs are provided within the submission file. Make sure to familarise yourself with the assignment brief before.

# Libraries

In [ ]:
# Install required packages
!pip install fsspec==2025.3.0
!pip install -q transformers[torch] datasets
!pip install -q evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


# All Pre-requsites (Hugging Face login, data import etc.)

Note: If you use a HuggingFace model, please do not share your access key, make sure all the outputs are visible, you will be marked down if outputs are not provided)

Provide all functions that you might need for processing the data and using the model (refer to Unit 4 tutorial for some hints)

In [ ]:
# Imports
import evaluate
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from tabulate import tabulate

In [ ]:
# Hugging Face login without sharing access key
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# Initialize the t5-small model, tokenizer and evaluator
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")
tokenizer = AutoTokenizer.from_pretrained("t5-small")
rouge = evaluate.load("rouge")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# Initialize variable

# Summaries (respond of the LLM)
summaries_zeroShot = []
summaries_fewShot = []
summaries_zeroShotPrompt = []
summaries_fewShotPrompt = []

# Dataset
universe = load_dataset("EdinburghNLP/xsum")

# Select 100 random data form Data Splits "test" for keep the execution time manageable
test_data = universe["test"].shuffle(seed=42).select(range(100))

# Extract the last examples of "test" Data Splits
max = len(test_data)
exampleTaken = 3 # >1 "overflow" the model (512 token limit) gives error
few_shot_input = "" # Hold the last examples in input format

few_shot_examples = test_data.select(range(max - exampleTaken, max))

for i in range(len(few_shot_examples)):
    few_shot_input += (
        f"document: {few_shot_examples["document"][i]}\n"
        f"summary: {few_shot_examples["summary"][i]}\n"
    )

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/300M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/204045 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11332 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11334 [00:00<?, ? examples/s]

In [ ]:
# Inspect the DataSet structure
universe

DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 204045
    })
    validation: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11332
    })
    test: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11334
    })
})

In [ ]:
# Inspect the structure of the segment "test" of the Data Splits
test_data

Dataset({
    features: ['document', 'summary', 'id'],
    num_rows: 100
})

In [ ]:
# Inspect 1º element of "test" Data Splits
test_data[0]

{'document': 'Sarah Johnson was one of 21 women heading to Liverpool when their minibus was hit by a lorry on the M62.\nHer friend Bethany Jones, 18, was killed while Ms Johnson and several others were badly hurt.\nMinibus driver James Johnson was jailed for more than six years for causing Bethany\'s death, in April 2013.\nMs Johnson, who broke her shoulder, back and pelvis, said the help she received from a charity while in hospital led her to want to support others.\nSpeaking publicly for the first time about the crash, Ms Johnson described how everyone was "excited and giddy" for the hen party.\n"To me the impact was just a massive explosion," she said.  "I thought the bus had blown up.\n"I remember the bus dropping on its side. The next thing, I woke up on the roadside so I\'d actually come out of the window."\nMs Johnson was taken to Leeds General Infirmary where she, along with Bethany\'s sister Amy Firth, underwent major surgery and spent time in intensive care.\nWhilst she was 

In [ ]:
# Inspect the examples extracted from the last part of "test" Data Splits
for i in range(len(few_shot_examples)):
    print(f"--- Example {i} ---")
    print(f"Document: {few_shot_examples["document"][i]}\n")
    print(f"Summary: {few_shot_examples["summary"][i]}\n")

--- Example 0 ---
Document: The 22-year-old Scotland international made five appearances for Leeds in 2012 and was released by the Rhinos at the end of the 2013 Super League season.
Hood will join his former Leeds team-mate Ben Jones-Bishop at the Red Devils next season.
"I'm very pleased that I've got this chance again and I'm going to take it with both hands," said Hood.
He told BBC Radio Manchester: "It's a massive challenge. I've had a couple of years out of top-flight rugby and it's a big change.
"You don't realise how much it benefits you being full-time. I'm really looking forward to getting back into it."

Summary: Salford Red Devils have signed hooker Liam Hood from Championship One Hunslet Hawks on a two-year deal from 2015.

--- Example 1 ---
Document: Anatoly Kucherena told reporters his client would remain in the transit zone at Moscow's Sheremetyevo airport, where he has been for the past month.
Earlier, airport officials said that Mr Kucherena had given Mr Snowden the tr

# Zero-shot Strategy

In [ ]:
# Give 0 example of the task asked before execution

# Loop for each dataset entry on the Data Splits "test"
for index in test_data:
    # Extract the "document" field of the Data Split
    document = index["document"]

    # Generated the input in text
    input_text = "summarize: " + document

    # Tokenize the input
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512, # 512 is the maximum sequence length for the model
        truncation=True
    )

    # Generate the summary in token format
    output_tokens = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=100, # 100 to avoids cutting sentences
        num_beams=4,
        early_stopping=False
    )

    # Decode the generated token back to text
    summary = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

    # Add the result on the variable
    summaries_zeroShot.append(summary)

In [ ]:
# Inspect the total summaries
print(f"Total summaries: {len(summaries_zeroShot)}")

Total summaries: 100


In [ ]:
# Inspect the 3 firsts results
for i,index in enumerate(test_data.select(range(3))):
    print(f"--- Example {i} ---")
    print(f"Document: {test_data[i]["document"]}\n")
    print(f"Summary: {summaries_zeroShot[i]}\n")

--- Example 0 ---
Document: Sarah Johnson was one of 21 women heading to Liverpool when their minibus was hit by a lorry on the M62.
Her friend Bethany Jones, 18, was killed while Ms Johnson and several others were badly hurt.
Minibus driver James Johnson was jailed for more than six years for causing Bethany's death, in April 2013.
Ms Johnson, who broke her shoulder, back and pelvis, said the help she received from a charity while in hospital led her to want to support others.
Speaking publicly for the first time about the crash, Ms Johnson described how everyone was "excited and giddy" for the hen party.
"To me the impact was just a massive explosion," she said.  "I thought the bus had blown up.
"I remember the bus dropping on its side. The next thing, I woke up on the roadside so I'd actually come out of the window."
Ms Johnson was taken to Leeds General Infirmary where she, along with Bethany's sister Amy Firth, underwent major surgery and spent time in intensive care.
Whilst she w

# Few-shot Strategy

In [ ]:
# Give few example of the task asked before execution

# Loop for the length (max) Data Splits "test" except the last examples
for i in range(max - exampleTaken):
    # Extract the "document" field of the Data Split
    document = test_data[i]["document"]

    # Generated the input in text
    input_text = few_shot_input + f"document: {document}\nsummary:"

    # Tokenize the input
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    # Generate the summary in token format
    output_tokens = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=100, # 100 to avoids cutting sentences
        num_beams=4,
        early_stopping=False
    )

    # Decode the generated token back to text
    summary = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

    # Add the result on the variable
    summaries_fewShot.append(summary)

In [ ]:
# Inspect the summaries
print(f"Total summaries: {len(summaries_fewShot)}")

Total summaries: 97


In [ ]:
# Print 3 first examples
for i,index in enumerate(test_data.select(range(3))):
    print(f"--- Example {i} ---")
    print(f"Document: {test_data[i]["document"]}\n")
    print(f"Summary: {summaries_fewShot[i]}\n")

--- Example 0 ---
Document: Sarah Johnson was one of 21 women heading to Liverpool when their minibus was hit by a lorry on the M62.
Her friend Bethany Jones, 18, was killed while Ms Johnson and several others were badly hurt.
Minibus driver James Johnson was jailed for more than six years for causing Bethany's death, in April 2013.
Ms Johnson, who broke her shoulder, back and pelvis, said the help she received from a charity while in hospital led her to want to support others.
Speaking publicly for the first time about the crash, Ms Johnson described how everyone was "excited and giddy" for the hen party.
"To me the impact was just a massive explosion," she said.  "I thought the bus had blown up.
"I remember the bus dropping on its side. The next thing, I woke up on the roadside so I'd actually come out of the window."
Ms Johnson was taken to Leeds General Infirmary where she, along with Bethany's sister Amy Firth, underwent major surgery and spent time in intensive care.
Whilst she w

# Prompt Engineering

## Zero-shot

In [ ]:
# Give 0 example of the task asked before execution, but with especific prompt

# Loop for each dataset entry on the Data Splits "test"
for index in test_data:
    # Extract the "document" field of the Data Split
    document = index["document"]

    # Generated the input in text
    input_text = "summarize the following news article in one concise sentence focusing on the main event: " + document

    # Tokenize the input
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512, # 512 is the maximum sequence length for the model
        truncation=True
    )

    # Generate the summary in token format
    output_tokens = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=100, # 100 to avoids cutting sentences
        num_beams=4,
        early_stopping=False
    )

    # Decode the generated token back to text
    summary = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

    # Add the result on the variable
    summaries_zeroShotPrompt.append(summary)

In [ ]:
# Inspect the summaries
print(f"Total summaries: {len(summaries_zeroShotPrompt)}")

Total summaries: 100


In [ ]:
# Inspect the 3 firsts results
for i,index in enumerate(test_data.select(range(3))):
    print(f"--- Example {i} ---")
    print(f"Document: {test_data[i]["document"]}\n")
    print(f"Summary: {summaries_zeroShotPrompt[i]}\n")

--- Example 0 ---
Document: Sarah Johnson was one of 21 women heading to Liverpool when their minibus was hit by a lorry on the M62.
Her friend Bethany Jones, 18, was killed while Ms Johnson and several others were badly hurt.
Minibus driver James Johnson was jailed for more than six years for causing Bethany's death, in April 2013.
Ms Johnson, who broke her shoulder, back and pelvis, said the help she received from a charity while in hospital led her to want to support others.
Speaking publicly for the first time about the crash, Ms Johnson described how everyone was "excited and giddy" for the hen party.
"To me the impact was just a massive explosion," she said.  "I thought the bus had blown up.
"I remember the bus dropping on its side. The next thing, I woke up on the roadside so I'd actually come out of the window."
Ms Johnson was taken to Leeds General Infirmary where she, along with Bethany's sister Amy Firth, underwent major surgery and spent time in intensive care.
Whilst she w

## Few-shot

In [ ]:
# Give 0 example of the task asked before execution, but with especific prompt

# Loop for the length (max) Data Splits "test" except the last example
for i in range(max - exampleTaken):
    # Extract the "document" field of the Data Split
    document = test_data[i]["document"]

    # Generated the input in text
    input_text = few_shot_input + "summarize the following news article in one concise sentence focusing on the main event:" + document

    # Tokenize the input
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    # Generate the summary in token format
    output_tokens = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=100, # 100 to avoids cutting sentences
        num_beams=4,
        early_stopping=False
    )

    # Decode the generated token back to text
    summary = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

    # Add the result on the variable
    summaries_fewShotPrompt.append(summary)

In [ ]:
# Inspect the summaries
print(f"Total summaries: {len(summaries_fewShotPrompt)}")

Total summaries: 97


In [ ]:
# Print 3 first examples
for i,index in enumerate(test_data.select(range(3))):
    print(f"--- Example {i} ---")
    print(f"Document: {test_data[i]["document"]}\n")
    print(f"Summary: {summaries_fewShotPrompt[i]}\n")

--- Example 0 ---
Document: Sarah Johnson was one of 21 women heading to Liverpool when their minibus was hit by a lorry on the M62.
Her friend Bethany Jones, 18, was killed while Ms Johnson and several others were badly hurt.
Minibus driver James Johnson was jailed for more than six years for causing Bethany's death, in April 2013.
Ms Johnson, who broke her shoulder, back and pelvis, said the help she received from a charity while in hospital led her to want to support others.
Speaking publicly for the first time about the crash, Ms Johnson described how everyone was "excited and giddy" for the hen party.
"To me the impact was just a massive explosion," she said.  "I thought the bus had blown up.
"I remember the bus dropping on its side. The next thing, I woke up on the roadside so I'd actually come out of the window."
Ms Johnson was taken to Leeds General Infirmary where she, along with Bethany's sister Amy Firth, underwent major surgery and spent time in intensive care.
Whilst she w

# Evaluation

In [ ]:
# Extract every "summary" field of "test_data" for comparison
references_all = [dataSet["summary"] for dataSet in test_data]
references_few_shot = [dataSet["summary"] for dataSet in test_data.select(range(max - exampleTaken))]

In [ ]:
# Zero-shot
rouge_zero = rouge.compute(
    predictions=summaries_zeroShot,
    references=references_all,
    use_stemmer=True
)

# Few-shot
rouge_few = rouge.compute(
    predictions=summaries_fewShot,
    references=references_few_shot,
    use_stemmer=True
)

# Zero-shot (Prompt)
rouge_zeroPrompt = rouge.compute(
    predictions=summaries_zeroShotPrompt,
    references=references_all,
    use_stemmer=True
)

# Few-shot (Prompt)
rouge_fewPrompt = rouge.compute(
    predictions=summaries_fewShotPrompt,
    references=references_few_shot,
    use_stemmer=True
)

In [ ]:
# Inpect the result of ROGUE in a table format
titles= ["Method","ROUGE-1", "ROUGE-2", "ROUGE-L"]
table_data = [
  ["Zero-shot",rouge_zero["rouge1"], rouge_zero["rouge2"], rouge_zero["rougeL"]],
  ["Few-shot",rouge_few["rouge1"],rouge_few['rouge2'],rouge_few['rougeL']],
  ["Zero-shot (Prompt)",rouge_zeroPrompt["rouge1"],rouge_zeroPrompt["rouge2"],rouge_zeroPrompt["rougeL"]],
  ["Few-shot (Prompt)",rouge_fewPrompt["rouge1"],rouge_fewPrompt["rouge2"],rouge_fewPrompt["rougeL"]]
]

print(tabulate(table_data, headers=titles))

Method                ROUGE-1     ROUGE-2    ROUGE-L
------------------  ---------  ----------  ---------
Zero-shot           0.190239   0.0268875    0.131721
Few-shot            0.0940397  0.00742379   0.075554
Zero-shot (Prompt)  0.19305    0.0250194    0.130785
Few-shot (Prompt)   0.0940397  0.00742379   0.075554


# Results Interpretation (max 200 words)

Note: If you reference something from the model outputs, make sure it can be traced (e.g., if you noticed a interesting pattern in the results, make sure to provide an example of that in the outputs).

**Findings: Did any method perform significantly better?**

Analysing the ROUGE metric shows that the T5-small model performed poorly overall, with scores ranging from 0.007 to 0.194 (where 1.0 indicates perfect performance). Despite being designed for text-to-text tasks, it appears insufficient for this dataset, likely due to its limited input capacity, 512 tokens. This suggests the model is not well-suited for handling large or complex inputs.
Among the tested methods, the best-performing approach was zero-shot with prompt engineering (ROUGE: 0.193099). This aligns with the model’s design, which relies on simple prefix-based tasks rather than in-context learning. Adding examples likely introduced noise rather than improving performance.

**Surprising errors: Any unexpected outputs or failure cases?**

The few-shot strategy generates an unexpected output when adding more than 1 example.  When adding 3 examples, it gives a random identical output for all cases; and when adding 5, you get a failure case where the output is false. Due to the model’s token limit, which caused truncation of inputs and effectively reduced few-shot prompts to one-shot or half articles for processes.

**Benchmark vs actual performance: Did results align with expectations?**

Results did not match expectations. It was anticipated that adding examples would improve performance; however, the opposite occurred. This can be explained by the model’s lack of instruction tuning, making it unable to effectively use additional context.